## Setup


In [1]:
import importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

if IN_COLAB:
    !sudo apt update
    !sudo apt install -y pciutils
    !sudo apt-get install zstd
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("Not running in Colab — skipping apt-based system setup.")
    print("Make sure Ollama is installed locally: https://ollama.com")

Not running in Colab — skipping apt-based system setup.
Make sure Ollama is installed locally: https://ollama.com


In [2]:
import subprocess
import time
import urllib.request


def ollama_running(host: str = "http://localhost:11434") -> bool:
    try:
        urllib.request.urlopen(host, timeout=1)
        return True
    except Exception:
        return False


if ollama_running():
    print("Ollama server already running.")
else:
    # Start the server as a background process
    process = subprocess.Popen("ollama serve", shell=True)
    # Wait a few seconds for the server to initialize
    time.sleep(5)

Ollama server already running.


In [3]:
!ollama pull llama3.2

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest 
pulling dde5aa3fc5ff: 100% ▕██████████████████▏ 2.0 GB                         
pulling 966de95ca8a6: 100% ▕██████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da: 100% ▕██████████████████▏ 7.7 KB                         
pulling a70ff7e570d9: 100% ▕██████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 34bb5ab01051: 100% ▕██████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [4]:
!pip install chromadb


You should consider upgrading via the '/Users/yasminkabir/Documents/GitHub/RAG_JET_Lab/venv/bin/python3 -m pip install --upgrade pip' command.


In [5]:
#import os
import chromadb
#import requests
from openai import OpenAI
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from PyPDF2 import PdfReader
#from google.colab import userdata
#from google import genai
from sentence_transformers import CrossEncoder

/Users/yasminkabir/Documents/GitHub/RAG_JET_Lab/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/yasminkabir/Documents/GitHub/RAG_JET_Lab/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Knowledge Base:

### Loading previous knowledge base:
-if loading previous knowledge base, just run the following cell   
-if building knowledge base, run all cells in this section

In [6]:
# chroma_client = chromadb.PersistentClient(path='/content/')
# collection = client.create_collection(
#     name="my_collection",
#     embedding_function=OpenAIEmbeddingFunction(
#         model_name="text-embedding-3-small"
#         api_key_env_var=OPENAI_API_KEY
#     )
# )

# NOTE: separate collection name so this experiment doesn't overwrite the
# main "test_collection" used by advanced_rag.ipynb / the app.
chroma_client = chromadb.PersistentClient(path='./chroma_db')
collection = chroma_client.get_or_create_collection(name="chunking_experiments")

In [7]:
def get_text_txt_md(file_path: str) -> str:
  with open(file_path, "r", encoding="utf-8") as f:
    text = f.read()
  return text

def get_text_pdf(pdf_path: str) -> str:
    """Extract raw text from a PDF file."""
    try:
        reader = PdfReader(pdf_path)
        return " ".join(page.extract_text() for page in reader.pages if page.extract_text())
    except Exception as e:
        raise RuntimeError(f"Error reading PDF: {e}")

# Chunking

Helper functions

In [8]:
import re
from transformers import AutoTokenizer

# Chroma's default embedding function is all-MiniLM-L6-v2. Its tokenizer
# reports a 512 max_length, but the model itself silently truncates beyond
# 256 word-piece tokens (per the model card) -- size chunks against that
# real limit, not the tokenizer's nominal one.
_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
EMBEDDING_MAX_TOKENS = 256
MIN_SENTENCE_TOKENS = 4


def count_tokens(text: str) -> int:
    return len(_tokenizer.encode(text, add_special_tokens=False)) #how many tokens in this string


def split_sentences(text: str) -> list[str]:
    """Split text into sentences using punctuation-based rules."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s for s in sentences if s]
#splits sentences based on . ! and ?

def _merge_short_fragments(sentences: list[str], min_tokens: int = MIN_SENTENCE_TOKENS) -> list[str]:
    """Merge fragments too short to be a real sentence on their own into a
    neighboring one. Citation-list text (e.g. "AfDB." "2016." "Airbnb.")
    trips the period-based splitter into treating every field as its own
    sentence -- left alone, that pollutes chunks with disconnected junk.
    """
    merged = []
    buffer = ""
    for sentence in sentences:
        buffer = f"{buffer} {sentence}".strip() if buffer else sentence
        if count_tokens(buffer) >= min_tokens:
            merged.append(buffer)
            buffer = ""
    if buffer:
        if merged:
            merged[-1] = f"{merged[-1]} {buffer}"
        else:
            merged.append(buffer)
    return merged
# makes sure that short "sentences" aren't counted as their own chunks, but are merged into neighboring sentences instead

def _hard_split(sentence: str, chunk_size: int) -> list[str]:
    """Fallback for a single 'sentence' that's already too long on its own
    (e.g. PDF extraction glued text together with sparse punctuation) --
    force-split it into chunk_size-token pieces so nothing downstream ever
    exceeds the embedding model's real limit.
    """
    ids = _tokenizer.encode(sentence, add_special_tokens=False)
    return [_tokenizer.decode(ids[i:i + chunk_size]) for i in range(0, len(ids), chunk_size)]
#splits a long sentence into smaller chunks if it exceeds the chunk limit

In [9]:

def chunk_text(text: str, chunk_size: int = 200, overlap: int = 25) -> list[str]:
    """Split text into chunks along sentence boundaries, sized in embedding
    tokens (not words).

    Sentences are accumulated until adding the next one would exceed
    `chunk_size` tokens; the last `overlap` tokens of a finished chunk are
    carried into the start of the next one for context continuity.
    `chunk_size` is capped at EMBEDDING_MAX_TOKENS, tiny fragments get
    merged into neighbors before packing, and any individual sentence still
    longer than the cap gets hard-split -- so no chunk can ever exceed what
    the embedding model actually encodes, and no chunk gets padded with
    disconnected junk fragments either.
    """
    chunk_size = min(chunk_size, EMBEDDING_MAX_TOKENS) #right now it is 256

    sentences = []
    for sentence in _merge_short_fragments(split_sentences(text)):
        if count_tokens(sentence) > chunk_size:
            sentences.extend(_hard_split(sentence, chunk_size))
        else:
            sentences.append(sentence)

    chunks = []
    current_sentences, current_len = [], 0

    for sentence in sentences:
        sentence_len = count_tokens(sentence)
        if current_sentences and current_len + sentence_len > chunk_size:
            chunk_str = " ".join(current_sentences)
            chunks.append(chunk_str)
            tail_ids = _tokenizer.encode(chunk_str, add_special_tokens=False)[-overlap:]
            overlap_text = _tokenizer.decode(tail_ids) if tail_ids else ""
            current_sentences = [overlap_text] if overlap_text else []
            current_len = len(tail_ids)
        current_sentences.append(sentence)
        current_len += sentence_len

    if current_sentences:
        chunks.append(" ".join(current_sentences))
    return chunks

# Building Corpus + Loading Database

### Metadata of References

In [ ]:
FILE_METADATA = {
    'rag_data/Digital entrepreneurship in Africa _ Brookings.pdf': {
        'title': 'Digital entrepreneurship in Africa',
        'authors': 'Afua Osei',
        'publisher': 'Brookings',
        'date': '2024-08-01',
        'type': 'report',
        #add trascript, report, etc. if available
    },
    'rag_data/Emerging-technology-policies-and-democracy-in-Africa.pdf': {
        'title': 'Emerging technology policies and democracy in Africa: South Africa, Kenya, Nigeria, Ghana, and Zambia in focus',
        'authors': 'Ayantola Alayande, Samuel Segun, Leah Junck',
        'publisher': 'Atlantic Council (Digital Forensic Research Lab)',
        'date': '2025-03',
        'type': 'report',
    },
    'rag_data/The African Tech Startups Funding Report 2025 (1).pdf': {
        'title': 'The African Tech Startups Funding Report 2025',
        'authors': 'Disrupt Africa',
        'publisher': 'Disrupt Africa',
        'date': '2025',
        'type': 'report',
    },
    'rag_data/venture-capital-and-the-rise-of-africa-s-tech-startups.pdf': {
        'title': "Venture Capital and the Rise of Africa's Tech Startups",
        'authors': 'Laurien Field, Marcio Cruz, Mariana Pereira-López, David Harrison',
        'publisher': 'IFC Economics and Market Research Department',
        'date': '2025-05',
        'type': 'report',
    },
}

In [11]:
files_paths = list(FILE_METADATA.keys())
chunks = []
metadatas = []
text = ""
for file in files_paths:
  if "pdf" in file:
    text = get_text_pdf(file)
  else:
    text = get_text_txt_md(file)
  file_chunks = chunk_text(text)
  chunks += file_chunks
  metadatas += [{"source": file, **FILE_METADATA[file]} for _ in file_chunks]

collection.upsert(
    documents=chunks,
    metadatas=metadatas,
    ids=[f"id{i}" for i in range(len(chunks))]
)

Token indices sequence length is longer than the specified maximum sequence length for this model (524 > 512). Running this sequence through the model will result in indexing errors


In [12]:
#test retrieve:
collection.query(
      query_texts=["What can you say about technology policies in Africa?"],
      n_results=4
  )

{'ids': [['id77', 'id45', 'id115', 'id47']],
 'embeddings': None,
 'documents': [['tackle the digital divide in the country first before it can concentrate on robust policy developments across all technology areas. 1. 3. Trends in emerging technologies research \nPolicy discourses on emerging technologies in Africa have lar-\ngely focused on their potential to drive industrialization, innova-\ntion, green transition, and economic growth on the continent.18 \nResearch on the impact of the technologies is also replete with \nan economic framing, especially their potential to accelerate \nAfrica’s industrialization ambitions.19 A few works have been \ndone on emerging technology use in critical sectors such \nas agriculture,20 education,21 energy,22 and healthcare.23 One \nconsistent finding from this body of research is that twenty-\nfirst century Africa is witnessing a growing adoption of forms \nof technologies more rapidly than it did in the postindepen-\ndence era.24 Such rapid adopt

## Chunking strategy evaluation

Retrieval-only hit-rate test. For each query below: does the expected
source file show up in the top-k results (`source_hit`), and does a known
keyword from the true answer show up in the retrieved chunk *text*
(`keyword_hit` — the stricter check, since it catches chunk boundaries
that separated the right file from the actual answer-bearing sentence)?

Re-run this after changing `chunk_text()` and re-ingesting (cell 11) to
compare strategies. Pass `rr_model` (loaded later, in the Generation
section) to also score hit-rate after reranking to `kp`.

In [13]:
EVAL_QUERIES = [
    {
        "query": "How much did Airbnb inject into South Africa's economy through job creation in 2022?",
        "expected_source": "rag_data/Digital entrepreneurship in Africa _ Brookings.pdf",
        "keywords": ["1.2 billion", "Airbnb"],
    },
    {
        "query": "Which organization published the report on emerging technology policies and democracy in Africa, and who funded it?",
        "expected_source": "rag_data/Emerging-technology-policies-and-democracy-in-Africa.pdf",
        "keywords": ["Atlantic Council", "Denmark"],
    },
    {
        "query": "How much funding did African tech startups raise in total in 2015, according to the first edition of the funding report?",
        "expected_source": "rag_data/The African Tech Startups Funding Report 2025 (1).pdf",
        "keywords": ["185,785,500", "Disrupt Africa"],
    },
    {
        "query": "According to Pitchbook data, how many African tech startups received funding annually by 2022, and how did that change by 2024?",
        "expected_source": "rag_data/venture-capital-and-the-rise-of-africa-s-tech-startups.pdf",
        "keywords": ["Pitchbook", "more than 700", "fewer than 400"],
    },
    {
        "query": "Which African cities are highlighted as leading tech startup hubs?",
        "expected_source": "rag_data/venture-capital-and-the-rise-of-africa-s-tech-startups.pdf",
        "keywords": ["Lagos", "Nairobi", "Cairo"],
    },
]


def evaluate_chunking(collection, eval_queries, k=10, kp=3, rr_model=None):
    """Retrieval-only hit-rate test for a chunking strategy.

    For each query, checks whether the expected source file appears among
    the top-k retrieved chunks (source_hit) and whether a known keyword from
    the true answer appears in the retrieved chunk text (keyword_hit --
    the stricter check, since it catches chunk boundaries that separated
    the right file from the actual answer-bearing sentence).

    If rr_model is given, also reranks the top-k down to top-kp and scores
    the same two checks post-rerank.
    """
    results = []
    for item in eval_queries:
        res = collection.query(query_texts=[item["query"]], n_results=k)
        docs = res["documents"][0]
        metas = res["metadatas"][0]

        source_hit = any(m.get("source") == item["expected_source"] for m in metas)
        keyword_hit = any(
            any(kw.lower() in doc.lower() for kw in item["keywords"]) for doc in docs
        )

        rerank_source_hit = None
        rerank_keyword_hit = None
        if rr_model is not None and docs:
            reranked = rr_model.rank(item["query"], docs, return_documents=False, top_k=kp)
            top_metas = [metas[r["corpus_id"]] for r in reranked]
            top_docs = [docs[r["corpus_id"]] for r in reranked]
            rerank_source_hit = any(m.get("source") == item["expected_source"] for m in top_metas)
            rerank_keyword_hit = any(
                any(kw.lower() in doc.lower() for kw in item["keywords"]) for doc in top_docs
            )

        results.append({
            "query": item["query"],
            "source_hit@k": source_hit,
            "keyword_hit@k": keyword_hit,
            "source_hit@kp": rerank_source_hit,
            "keyword_hit@kp": rerank_keyword_hit,
        })

    n = len(results)
    print(f"source_hit@{k}:  {sum(r['source_hit@k'] for r in results)}/{n}")
    print(f"keyword_hit@{k}: {sum(r['keyword_hit@k'] for r in results)}/{n}")
    if rr_model is not None:
        print(f"source_hit@{kp} (reranked):  {sum(bool(r['source_hit@kp']) for r in results)}/{n}")
        print(f"keyword_hit@{kp} (reranked): {sum(bool(r['keyword_hit@kp']) for r in results)}/{n}")
    for r in results:
        print(r)

    return results

In [14]:
evaluate_chunking(collection, EVAL_QUERIES, k=10)

source_hit@10:  5/5
keyword_hit@10: 5/5
{'query': "How much did Airbnb inject into South Africa's economy through job creation in 2022?", 'source_hit@k': True, 'keyword_hit@k': True, 'source_hit@kp': None, 'keyword_hit@kp': None}
{'query': 'Which organization published the report on emerging technology policies and democracy in Africa, and who funded it?', 'source_hit@k': True, 'keyword_hit@k': True, 'source_hit@kp': None, 'keyword_hit@kp': None}
{'query': 'How much funding did African tech startups raise in total in 2015, according to the first edition of the funding report?', 'source_hit@k': True, 'keyword_hit@k': True, 'source_hit@kp': None, 'keyword_hit@kp': None}
{'query': 'According to Pitchbook data, how many African tech startups received funding annually by 2022, and how did that change by 2024?', 'source_hit@k': True, 'keyword_hit@k': True, 'source_hit@kp': None, 'keyword_hit@kp': None}
{'query': 'Which African cities are highlighted as leading tech startup hubs?', 'source_hi

[{'query': "How much did Airbnb inject into South Africa's economy through job creation in 2022?",
  'source_hit@k': True,
  'keyword_hit@k': True,
  'source_hit@kp': None,
  'keyword_hit@kp': None},
 {'query': 'Which organization published the report on emerging technology policies and democracy in Africa, and who funded it?',
  'source_hit@k': True,
  'keyword_hit@k': True,
  'source_hit@kp': None,
  'keyword_hit@kp': None},
 {'query': 'How much funding did African tech startups raise in total in 2015, according to the first edition of the funding report?',
  'source_hit@k': True,
  'keyword_hit@k': True,
  'source_hit@kp': None,
  'keyword_hit@kp': None},
 {'query': 'According to Pitchbook data, how many African tech startups received funding annually by 2022, and how did that change by 2024?',
  'source_hit@k': True,
  'keyword_hit@k': True,
  'source_hit@kp': None,
  'keyword_hit@kp': None},
 {'query': 'Which African cities are highlighted as leading tech startup hubs?',
  'source

# Generation:

In [ ]:
#loaded models
client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)
model = "llama3.2"
rr_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
query = "When was the snake game made?"

In [ ]:
def call_llm(model: str, prompt: str, client):
  try:
    response = client.responses.create(
      model=model,
      input=prompt,
    )
    return response.output_text
  except Exception as e:
      raise RuntimeError(f"Ollama LLM query failed: {e}")

In [ ]:
def filter_chunks(chunks, metadatas, query, rr_model, kp):
    """Rerank and filter chunks based on relevance to the query."""
    reranked = rr_model.rank(query, chunks, return_documents=False, top_k=kp)
    print(reranked)
    # for item in kp_chunks:
    #   print(item)
    return ['Source: ' + metadatas[item['corpus_id']]['source'] + '\n' + chunks[item['corpus_id']] for item in reranked]

In [ ]:
def gen_response(query, model, client, rr_model, collection, k, kp):
  res_k = collection.query(
      query_texts=[query],
      n_results=k
  )
  print(res_k['metadatas'])
  # for i, chunk in enumerate(res_k['documents'][0]):
  #   print(str(i) + ": " + chunk)
  
  kp_chunks = filter_chunks(res_k['documents'][0], res_k['metadatas'][0], query, rr_model, kp=3)
  context = "\n".join(kp_chunks)
  prompt = f"{query} Only use the following context to answer this question and cite the source you use in your answer. Clearly state when the context does not contain the answer: {context}"
  return call_llm(model, prompt, client)

In [ ]:
gen_response(query, model, client, rr_model, collection, 10, 3)